In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

df = pd.read_csv("Mental Health Dataset.csv")

df = df.drop(columns=["Timestamp"], errors="ignore")
df = df.drop_duplicates().reset_index(drop=True)

male_terms = {"male", "m", "man", "male (cis)", "cis male", "cis man"}
female_terms = {"female", "f", "woman", "female (cis)", "cis female", "cis woman"}

def clean_gender(g):
    s = str(g).strip().lower()
    if s in male_terms:
        return "Male"
    if s in female_terms:
        return "Female"
    return "Other"

df["Gender"] = df["Gender"].apply(clean_gender)

X = df.drop(columns=["treatment"])
y = df["treatment"].map({"Yes": 1, "No": 0})

cat_cols = X.select_dtypes(include="object").columns

preprocess = ColumnTransformer([
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        cat_cols
    )
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

models = {
    "LogReg": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = []

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocess),
        ("model", model)
    ])

    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    results.append([name, scores.mean(), scores.std()])

    print(
        f"{name:15s} AUC = {scores.mean():.4f} ± {scores.std():.4f}"
    )

results_df = pd.DataFrame(
    results,
    columns=["Model", "AUC", "STD"]
).sort_values("AUC", ascending=False)

print("\n")
print(results_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=results_df, x="AUC", y="Model")
plt.title("Model Comparison")
plt.tight_layout()
plt.show()

pipe = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE))
])

param_grid = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__class_weight": [None, "balanced"]
}

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_grid,
    n_iter=20,
    cv=cv,
    scoring="roc_auc",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

search.fit(X_train, y_train)

best_model = search.best_estimator_

print("\nBest ROC-AUC:", round(search.best_score_, 4))
print("Best Parameters:", search.best_params_)

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

comparison["Actual"] = comparison["Actual"].map(
    {1: "Treatment", 0: "No Treatment"}
)

comparison["Predicted"] = comparison["Predicted"].map(
    {1: "Treatment", 0: "No Treatment"}
)

print("\nActual vs Predicted")
print(comparison.head(20))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

comparison["Actual"].value_counts().plot(
    kind="bar",
    ax=axes[0]
)
axes[0].set_title("Actual")

comparison["Predicted"].value_counts().plot(
    kind="bar",
    ax=axes[1]
)
axes[1].set_title("Predicted")

plt.tight_layout()
plt.show()

print("\nAccuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall   :", round(recall_score(y_test, y_pred), 4))
print("F1 Score :", round(f1_score(y_test, y_pred), 4))
print("ROC AUC  :", round(roc_auc_score(y_test, y_prob), 4))

print("\nClassification Report\n")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["No Treatment", "Treatment"]
    )
)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Treatment", "Treatment"],
    yticklabels=["No Treatment", "Treatment"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_value = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {roc_value:.3f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.show()

perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": perm.importances_mean
}).sort_values(
    "Importance",
    ascending=False
).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=feature_importance,
    x="Importance",
    y="Feature"
)
plt.title("Top Feature Importance")
plt.tight_layout()
plt.show()

sample = pd.DataFrame([{
    "Gender": "Male",
    "Country": "Pakistan",
    "Occupation": "Student",
    "self_employed": "No",
    "family_history": "Yes",
    "Days_Indoors": "1-14 days",
    "Growing_Stress": "Yes",
    "Changes_Habits": "Yes",
    "Mental_Health_History": "Yes",
    "Mood_Swings": "High",
    "Coping_Struggles": "Yes",
    "Work_Interest": "No",
    "Social_Weakness": "Yes",
    "mental_health_interview": "No",
    "care_options": "Yes"
}])

sample["Gender"] = sample["Gender"].apply(clean_gender)

probability = best_model.predict_proba(sample)[0, 1]
prediction = best_model.predict(sample)[0]

print("\nSample Prediction")
print("Treatment" if prediction == 1 else "No Treatment")
print("Probability:", round(probability * 100, 2), "%")

FileNotFoundError: [Errno 2] No such file or directory: 'Mental Health Dataset.csv'